# RAG Sufficient Context — Google Colab pipeline

End-to-end runner for the *sufficient-context* selective RAG pipeline on a Colab GPU (free T4 or Colab Pro V100/A100).

**What it does**

1. Clones the public repo `SachaYT1/rag-sufficient-context` (main).
2. Installs the pinned dependencies plus optional `dense`/`gate` extras.
3. Runs one or more experiment configs (`run_experiment` / `run_matrix`).
4. Builds headline figures and prints the leaderboard.
5. Zips `results/` and `reports/figures/` and (optionally) copies them into Google Drive so you don't lose them when the runtime recycles.

**Setup in the Colab UI**

- Runtime → **Change runtime type → GPU = T4** (or V100/A100 if you have Colab Pro).
- Leave HIGH-RAM off unless you run Qwen-7B.
- (Optional) `HF_TOKEN` via the "key" icon in the left sidebar if you switch to Llama-3.1. Default Qwen/Mistral configs don't need it.


## 0. Sanity — GPU check

In [ ]:
!nvidia-smi || echo 'no GPU — change runtime type to GPU'
!df -h /content | tail -1
import os; print('cwd:', os.getcwd())

## 1. Clone the repo

In [ ]:
import os
REPO_URL = 'https://github.com/SachaYT1/rag-sufficient-context.git'
BRANCH = 'main'
WORKDIR = '/content/rag-sufficient-context'

if not os.path.exists(WORKDIR):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} {WORKDIR}
else:
    !cd {WORKDIR} && git pull --ff-only

%cd {WORKDIR}
!ls -la

## 2. Install dependencies

Colab images already ship with `torch`, `transformers`, `datasets`, `numpy`, `scikit-learn`, `matplotlib`. We top up only what is missing and install the project in editable mode so `from src ...` works.

In [ ]:
!pip install -q --upgrade pip
!pip install -q -e '.[dense,gate]'
!pip install -q rank-bm25 sentence-transformers xgboost
import torch, transformers, datasets, sklearn, numpy as np, matplotlib
print('torch', torch.__version__, 'cuda_available', torch.cuda.is_available())
print('transformers', transformers.__version__)
print('datasets', datasets.__version__)

## 3. (Optional) Mount Google Drive

Colab wipes `/content/` when the runtime recycles. Mounting Drive gives you a persistent path and lets you keep the HuggingFace cache across sessions.

In [ ]:
USE_DRIVE = True   # set to False if you don't want to mount Drive
DRIVE_RESULTS_DIR = '/content/drive/MyDrive/rag_sufficient_context_runs'
HF_CACHE_ON_DRIVE = '/content/drive/MyDrive/hf_cache'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
    os.makedirs(HF_CACHE_ON_DRIVE, exist_ok=True)
    os.environ['HF_HOME'] = HF_CACHE_ON_DRIVE
    os.environ['TRANSFORMERS_CACHE'] = HF_CACHE_ON_DRIVE
    os.environ['HF_DATASETS_CACHE'] = HF_CACHE_ON_DRIVE
    print('Drive mounted; HuggingFace cache ->', HF_CACHE_ON_DRIVE)
else:
    print('skipping Drive mount — results will live only in /content/')

## 4. (Optional) HuggingFace authentication

Only needed for gated models (Llama-3.1). Default Qwen/Mistral configs skip this.

Add `HF_TOKEN` via the "key" icon in the Colab sidebar, then uncomment below.

In [ ]:
# from google.colab import userdata
# from huggingface_hub import login
# login(token=userdata.get('HF_TOKEN'))
print('skipping HF login — using open-weights Qwen/Mistral only')

## 5. Pick experiment configs

Start with the fast one (`qwen3b_bm25`, ~20–40 min on T4, 500 HotPotQA examples). The hybrid/asymmetric configs need >=16 GB VRAM — use V100/A100 (Colab Pro) or swap the generator down to Qwen-3B.

In [ ]:
CONFIGS = [
    'configs/experiments/qwen3b_bm25.yaml',
    # 'configs/experiments/qwen7b_hybrid.yaml',   # V100/A100 only
    # 'configs/experiments/asymmetric.yaml',      # loads 2 models — A100
    # 'configs/experiments/baseline.yaml',        # Mistral-7B baseline
    # 'configs/experiments/nq_open.yaml',         # second dataset
]
print('will run:', CONFIGS)

## 6. Smoke test (20 examples)

Patches `num_examples` down so we can confirm the pipeline is green in ~3 minutes before committing to a full 500-example run.

In [ ]:
import yaml, pathlib

src_cfg = pathlib.Path(CONFIGS[0])
smoke_cfg = pathlib.Path('configs/experiments/smoke_colab.yaml')
data = yaml.safe_load(src_cfg.read_text())
data.setdefault('dataset', {})['num_examples'] = 20
data.setdefault('experiment', {})['name'] = 'smoke_colab'
smoke_cfg.write_text(yaml.safe_dump(data))
print('smoke config written:', smoke_cfg)
!python -m run.pipeline.run_experiment {smoke_cfg}

## 7. Full runs

In [ ]:
import subprocess, time
summaries = []
for cfg in CONFIGS:
    print(f'=== {cfg} ===')
    t0 = time.time()
    rc = subprocess.call(['python', '-m', 'run.pipeline.run_experiment', cfg])
    dt = time.time() - t0
    print(f'exit={rc}  wall={dt/60:.1f} min')
    summaries.append({'config': cfg, 'rc': rc, 'wall_min': dt / 60})
summaries

## 8. Aggregate — matrix leaderboard

In [ ]:
!python -m run.pipeline.run_matrix {' '.join(CONFIGS)} --output_dir results/matrix || echo 'matrix runner failed — per-experiment summaries are still intact'
!cat results/matrix/leaderboard.md 2>/dev/null || echo 'leaderboard.md not produced'

## 9. Headline figures

In [ ]:
!python -m run.pipeline.make_figures --matrix results/matrix/summary.json --results_root results --output_dir reports/figures
import glob; print(glob.glob('reports/figures/*'))

In [ ]:
from IPython.display import Image, display
for path in sorted(glob.glob('reports/figures/*.pdf')) + sorted(glob.glob('reports/figures/*.png')):
    print(path)
    if path.endswith('.png'):
        display(Image(path))

## 10. Statistical analysis — bootstrap CI + paired test

In [ ]:
import json, numpy as np
from pathlib import Path
from src.analysis import bootstrap_aurc_ci, paired_bootstrap_test, stratified_selective_curves

for cfg in CONFIGS:
    exp_name = Path(cfg).stem
    curves_path = Path('results') / exp_name / 'selective_curves.json'
    eval_path = Path('results') / exp_name / 'evaluation.json'
    if not curves_path.exists():
        print('missing', curves_path)
        continue
    curves = json.loads(curves_path.read_text())
    eval_data = json.loads(eval_path.read_text())
    per_example = eval_data['per_example']
    total = len(per_example)
    b = np.asarray(curves['confidence_scores'], dtype=float)
    p = np.asarray(curves['gate_scores'], dtype=float)
    y = np.asarray(curves['labels'], dtype=float)
    print(f'--- {exp_name} (n_answered={len(y)}, total={total}) ---')
    print('Baseline AURC CI:', bootstrap_aurc_ci(b, y, total=total, n_bootstrap=500))
    print('Proposed AURC CI:', bootstrap_aurc_ci(p, y, total=total, n_bootstrap=500))
    print('Paired test     :', paired_bootstrap_test(b, p, y, total=total, n_bootstrap=500))
    strat = stratified_selective_curves(per_example)
    print('Stratified keys :', list(strat))
    print()

## 11. Archive + persist results

Zips everything and (if Drive is mounted) copies the archives into `DRIVE_RESULTS_DIR` so the runtime disconnect can't eat your run.

In [ ]:
import shutil, datetime
stamp = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
archive_base = f'/content/rag_sufficient_context_{stamp}'
shutil.make_archive(archive_base, 'zip', WORKDIR, 'results')
print('results archived ->', archive_base + '.zip')

figures_base = f'/content/rag_sufficient_context_figures_{stamp}'
if os.path.exists('reports/figures'):
    shutil.make_archive(figures_base, 'zip', WORKDIR, 'reports/figures')
    print('figures archived ->', figures_base + '.zip')

if USE_DRIVE:
    for src in [archive_base + '.zip', figures_base + '.zip']:
        if os.path.exists(src):
            dst = os.path.join(DRIVE_RESULTS_DIR, os.path.basename(src))
            shutil.copy2(src, dst)
            print('copied to Drive ->', dst)
!ls -la /content | head

## 12. Download ZIPs directly (no Drive)

If you skipped the Drive step, trigger a browser download.

In [ ]:
from google.colab import files
if os.path.exists(archive_base + '.zip'):
    files.download(archive_base + '.zip')
if os.path.exists(figures_base + '.zip'):
    files.download(figures_base + '.zip')

## 13. Optional — interactive demo

Threshold slider with live coverage / selective accuracy / risk.

In [ ]:
from pathlib import Path
import json
exp_name = Path(CONFIGS[0]).stem
curves = json.loads((Path('results') / exp_name / 'selective_curves.json').read_text())
try:
    from src.demo import build_threshold_widget
    total = len(curves['labels'])
    build_threshold_widget(scores=curves['gate_scores'], labels=curves['labels'], total=total)
except ImportError as exc:
    print('ipywidgets not available in this kernel:', exc)